In [2]:
# Import libraries 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3

In [ ]:
#1. import database / data

In [6]:
con = sqlite3.connect('Drop_off_Analysis.db')

sql_query = """
        SELECT name
        FROM sqlite_master
        WHERE type='table'
"""

tables = pd.read_sql(sql_query, con)

# Create DataFrame for each table

for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", con)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")

con.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [8]:
# Print table names and column names :

con = sqlite3.connect('Drop_off_Analysis.db')

for table_name in tables['name']:
    print(f"\nTable name : {table_name}")

    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns=pd.read_sql(columns_query, con)
    print("Columns :")
    print(columns['name'].tolist())

con.close()




Table name : db_customer
Columns :
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table name : db_subscription
Columns :
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table name : db_support
Columns :
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [9]:
#2 DATA CLEANING

# Table 1 : df_db_customer
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [10]:
df_db_customer.info( )

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     object
 1   name        21 non-null     object
 2   country     18 non-null     object
 3   state       21 non-null     object
 4   gender      21 non-null     object
 5   dob         21 non-null     object
 6   interests   4 non-null      object
 7   pincode     0 non-null      object
dtypes: object(8)
memory usage: 1.4+ KB


In [47]:
# 1. DROP Columns - intrest, pincode
# df_db_customer.drop(df_db_customer.columns[-2:], axis=1)

# 2. Rename column - name to customer_name
# df_db_customer.rename(columns = {'name':'Customer_name'}, inplace=True)

# 3. Change dtype - DOB
# df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

# 4. data standardization - gender
# df_db_customer['gender'].unique() # Returns unique value of the col
# df_db_customer['gender'].replace({'Men':'Male','Women':'Female'}, inplace=True)

# 5. Fix Missing values - country
# df_db_customer[df_db_customer['country'].isna()]
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))
df_db_customer

,customerid,Customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05
5,0013-MHZWF,durga,India,Delhi,Female,1988-12-10
6,0013-SMEOE,mina,India,Meghalaya,Female,1976-09-21
7,0014-BMAQU,madan,India,Rajasthan,Male,1999-03-14
8,0015-UOCOJ,maya,Nepal,Kathmandu,Female,1985-07-07
9,0016-QLJIS,arjun,Nepal,Kathmandu,Male,1993-10-29


In [89]:
# Table 2 : 
df_db_subscription

# 1. Change dtype of : subscription_start_date,renewal_date, cancellation_date

#df_db_subscription['subscription_start_date'] = pd.to_datetime(df_db_subscription['subscription_start_date'])

#date_col = ['renewal_date', 'cancellation_date']
#df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)

# df_db_subscription.info()
# df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,drop_off_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88
5,0013-MHZWF,2022-06-18,Paid,2025-06-18,Standard,Annual,NaT,None,17.99,720,22
6,0013-SMEOE,2021-09-30,Refferal,2024-09-30,Basic,Monthly,2024-11-15,Not enough content,8.99,230,79
7,0014-BMAQU,2020-02-14,Organic,2025-02-14,Premium,Annual,NaT,None,22.99,1840,5
8,0015-UOCOJ,2023-07-22,Organic,2024-07-22,Standard,Monthly,NaT,None,13.99,240,34
9,0016-QLJIS,2022-04-03,Organic,2025-04-03,Basic,Annual,NaT,None,6.99,335,41


In [88]:
# Table 3 : db_support
# 1. Drop columnms : col_1, comment
# df_db_support = df_db_support.drop(df_db_support.columns[-2:], axis=1)

# 2. change dtype of : obj to datetime
# df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
# df_db_support.info()

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30
5,0017-IUDMW,2024-04-10,Y,25
6,0019-EFAEP,2024-09-27,Y,30
7,0022-TCJCI,2024-09-13,Y,10
8,0022-TCJCI,2024-09-14,N,90


In [90]:
# Feature Engineering and Data Analysis 

#create a new column using existing column
df_db_subscription['dropoff_flag'] = np.where(df_db_subscription['cancellation_reason'].notna(),1, 0)

In [91]:
df_db_subscription.head

<bound method NDFrame.head of     customerid subscription_start_date subscription_type renewal_date  \
0   0002-ORFBO              2021-03-15          Refferal   2025-03-15   
1   0003-MKNFE              2020-08-01              Paid   2024-08-01   
2   0004-TLHLJ              2022-11-20           Organic   2025-11-20   
3   0011-IGKFF              2019-05-10              Paid   2025-05-10   
4   0013-EXCHZ              2023-01-05          Refferal   2024-01-05   
5   0013-MHZWF              2022-06-18              Paid   2025-06-18   
6   0013-SMEOE              2021-09-30          Refferal   2024-09-30   
7   0014-BMAQU              2020-02-14           Organic   2025-02-14   
8   0015-UOCOJ              2023-07-22           Organic   2024-07-22   
9   0016-QLJIS              2022-04-03           Organic   2025-04-03   
10  0017-DINOC              2021-12-01           Organic   2025-12-01   
11  0017-IUDMW              2023-03-17          Refferal   2024-03-17   
12  0018-NYROU       

In [120]:
# fix support table with duplicates first then merge all tables with left join

In [122]:
df_db_subscription.shape

(21, 12)

In [121]:
df.shape

(21, 21)

In [108]:
df_db_subscription['customerid'].nunique()

21

In [109]:
df_db_customer['customerid'].nunique()

21

In [110]:
df_db_support['customerid'].nunique()

7

In [112]:
df_db_support['customerid'].size

9

In [114]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [119]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep='last')

In [ ]:
# merge 2 tables

df = ((df_db_subscription
    .merge(df_db_customer, on='customerid', how='left'))
    .merge(df_db_support, on='customerid', how='left'))

In [123]:
df.to_csv('exported_dropoff_data.csv', index=False)

In [ ]:
# Data Analysis 

In [124]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'drop_off_score',
       'dropoff_flag', 'Customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='object')

In [130]:
# 1. drop_off Rate

drop_off = df['dropoff_flag'].mean()*100
print(f"Drop-off Rate : {round(drop_off,2)}%")

Drop-off Rate : 28.57%


In [132]:
# 2. retention_rate

retention_rate = 100-drop_off
print(f"Retention Rate : {round(retention_rate,2)}%")

Retention Rate : 71.43%


In [135]:
# drop_off by Plan_type

drop_off_by_plan = df.groupby('plan_type')['dropoff_flag'].mean().mul(100).round(2).reset_index(name="dropoff_rate_percentage")
drop_off_by_plan

# maximum dropoff from basic plan

,plan_type,dropoff_rate_percentage
0,Basic,60.00
1,Premium,14.29
2,Standard,22.22


In [136]:
# dropoff rate statewise 

drop_off_by_state = df.groupby('state')['dropoff_flag'].mean().mul(100).round(2).reset_index(name="dropoff_rate_percentage")
drop_off_by_state['sum_of_rev', 'count_by_user']

,state,dropoff_rate_percentage
0,Delhi,25.00
1,Karnataka,100.00
2,Kathmandu,0.00
3,Maharashtra,0.00
4,Meghalaya,66.67
5,Nagaland,0.00
6,Rajasthan,0.00
7,Telangana,50.00
8,Uttar Pradesh,0.00
